In [1]:
import json

f = open('../../rag_utility/eval_results/short_answers_0shot_1calls_0_0_bm25_dl_nq_test_concise_eval.json')
zero_evals = json.load(f)
f.close()

_k = 3
_ret = 'mt5'
f = open(f'../../rag_utility/eval_results/short_answers_{_k}shot_1calls_1_0_{_ret}_dl_nq_test_concise_eval.json')
k_evals = json.load(f)
f.close()

In [2]:
import pandas as pd
import numpy as np

qids_column = list(zero_evals.keys())
zero_f1_column = [zero_evals[qid]['0']['0']['F1'] for qid in qids_column]
k_f1_column = [k_evals[qid]['0']['0']['F1'] for qid in qids_column]
zero_em_column = [zero_evals[qid]['0']['0']['EM'] for qid in qids_column]
k_em_column = [k_evals[qid]['0']['0']['EM'] for qid in qids_column]

eval_df = pd.DataFrame(np.array([zero_f1_column, k_f1_column, zero_em_column, k_em_column]).T, columns=['F1(0)', 'F1(k)', 'EM(0)', 'EM(k)'])
eval_df['qid'] = qids_column
eval_df['utility'] = eval_df['F1(k)'] - eval_df['F1(0)']

eval_df

,F1(0),F1(k),EM(0),EM(k),qid,utility
0,1.0,0.800000,1.0,0.0,test_0,-0.200000
1,0.0,0.000000,0.0,0.0,test_1,0.000000
2,0.0,0.000000,0.0,0.0,test_2,0.000000
3,0.0,0.000000,0.0,0.0,test_3,0.000000
4,0.0,0.000000,0.0,0.0,test_4,0.000000
...,...,...,...,...,...,...
3605,0.0,0.666667,0.0,0.0,test_3605,0.666667
3606,0.4,1.000000,0.0,1.0,test_3606,0.600000
3607,0.0,1.000000,0.0,1.0,test_3607,1.000000
3608,1.0,0.500000,1.0,0.0,test_3608,-0.500000


In [3]:
from tools import coherence_cal

In [4]:
from tools import matrix_tools

In [5]:
import pickle as pkl

res, _, doc_length_dict = coherence_cal.get_res_and_dicts('nq_test', _ret)


In [6]:
import math

f = open(f'../coherence_res/bi-directional/nq_test_3_{_ret}.pkl', 'rb')
matrix_book = pkl.load(f)
f.close()

w = 12
coh_dict = coherence_cal.cal_coherence(matrix_book, _k, doc_length_dict, w, math.ceil(w/2), bidirectional=0)
coh_df = pd.DataFrame(np.array([list(coh_dict.keys()), list(coh_dict.values())]).T, columns=['qid', 'coherence'])

In [7]:
final_df = eval_df.merge(coh_df, on='qid')
final_df.utility = final_df.utility.astype('float')
final_df.coherence = final_df.coherence.astype('float')
print(final_df.shape)
print(final_df['F1(k)'].mean())

from scipy import stats

print('Correlation with utility')
print('r', stats.pearsonr(final_df.coherence.values, final_df.utility.values))
print('rho', stats.spearmanr(final_df.coherence.values, final_df.utility.values))
print('tau', stats.kendalltau(final_df.coherence.values, final_df.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df.coherence.values, final_df['F1(k)'].values))
print('rho', stats.spearmanr(final_df.coherence.values, final_df['F1(k)'].values))
print('tau', stats.kendalltau(final_df.coherence.values, final_df['F1(k)'].values))

(3610, 7)
0.4227221393564884
Correlation with utility
r PearsonRResult(statistic=0.021173065110384963, pvalue=0.20342673798604408)
rho SignificanceResult(statistic=0.02623877842978851, pvalue=0.11497116131480689)
tau SignificanceResult(statistic=0.01925875647718901, pvalue=0.11464338897870248)

Correlation with F1
r PearsonRResult(statistic=0.05941484507985084, pvalue=0.0003545933616323084)
rho SignificanceResult(statistic=0.06473217192581003, pvalue=9.939818936816175e-05)
tau SignificanceResult(statistic=0.04918880043149799, pvalue=9.709863590476896e-05)


In [8]:
final_df

,F1(0),F1(k),EM(0),EM(k),qid,utility,coherence
0,1.0,0.800000,1.0,0.0,test_0,-0.200000,0.008936
1,0.0,0.000000,0.0,0.0,test_1,0.000000,0.005105
2,0.0,0.000000,0.0,0.0,test_2,0.000000,0.017625
3,0.0,0.000000,0.0,0.0,test_3,0.000000,0.019677
4,0.0,0.000000,0.0,0.0,test_4,0.000000,0.008997
...,...,...,...,...,...,...,...
3605,0.0,0.666667,0.0,0.0,test_3605,0.666667,0.008164
3606,0.4,1.000000,0.0,1.0,test_3606,0.600000,0.026283
3607,0.0,1.000000,0.0,1.0,test_3607,1.000000,0.027256
3608,1.0,0.500000,1.0,0.0,test_3608,-0.500000,0.063478


In [9]:
import qpp_methods

In [25]:
qpp = qpp_methods.QPP('nq_test')

22:54:24.927 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 660.3 MiB of memory would be required.


In [11]:
import pyterrier_dr

tct_model = pyterrier_dr.TctColBert()
# tct_index = pyterrier_dr.FlexIndex('/mnt/indices/msmarco-passage.tct-hnp.flex')
tct_index = pyterrier_dr.FlexIndex('/mnt/indices/nq_tct_colbert_index_1.flex')

In [12]:
# spatial_qpp_df = qpp.qpp_in_batch(res[res.qid.isin(res.qid.unique())], 'spatial', _k, tct_model, tct_index)

In [13]:
# spatial_qpp_df.to_csv(f'./qpp_cache_dense_{_ret}_{_k}.csv', index=False)

In [26]:
qpp_df = qpp.qpp_in_batch(res, 'nqc', _k, tct_model, tct_index)

In [27]:
final_df_1 = final_df.merge(qpp_df, on='qid')
final_df_1['coh_spatial'] = final_df_1['coherence'].apply(lambda x: math.log(1+x)) + final_df_1['qpp_estimate'].apply(lambda x: math.log(1+x))
final_df_1.shape

(3610, 11)

In [28]:
from scipy import stats

print('Correlation with utility')
print('r', stats.pearsonr(final_df_1.qpp_estimate.values, final_df_1.utility.values))
print('rho', stats.spearmanr(final_df_1.qpp_estimate.values, final_df_1.utility.values))
print('tau', stats.kendalltau(final_df_1.qpp_estimate.values, final_df_1.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df_1.qpp_estimate.values, final_df_1['F1(k)'].values))
print('rho', stats.spearmanr(final_df_1.qpp_estimate.values, final_df_1['F1(k)'].values))
print('tau', stats.kendalltau(final_df_1.qpp_estimate.values, final_df_1['F1(k)'].values))

Correlation with utility
r PearsonRResult(statistic=-0.034633839877145065, pvalue=0.037450557779870185)
rho SignificanceResult(statistic=-0.07551176859842594, pvalue=5.57472008122224e-06)
tau SignificanceResult(statistic=-0.05507474904384708, pvalue=6.431512623671548e-06)

Correlation with F1
r PearsonRResult(statistic=-0.0411655292573047, pvalue=0.013377743241936945)
rho SignificanceResult(statistic=-0.14676039900505722, pvalue=7.801592811483663e-19)
tau SignificanceResult(statistic=-0.11180352286870125, pvalue=8.048769379576529e-19)


In [29]:
from scipy import stats

print('Correlation with utility')
print('r', stats.pearsonr(final_df_1.coh_spatial.values, final_df_1.utility.values))
print('rho', stats.spearmanr(final_df_1.coh_spatial.values, final_df_1.utility.values))
print('tau', stats.kendalltau(final_df_1.coh_spatial.values, final_df_1.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df_1.coh_spatial.values, final_df_1['F1(k)'].values))
print('rho', stats.spearmanr(final_df_1.coh_spatial.values, final_df_1['F1(k)'].values))
print('tau', stats.kendalltau(final_df_1.coh_spatial.values, final_df_1['F1(k)'].values))

Correlation with utility
r PearsonRResult(statistic=-0.04138014875479073, pvalue=0.012902150492715682)
rho SignificanceResult(statistic=-0.037326271926717225, pvalue=0.024916844373359174)
tau SignificanceResult(statistic=-0.027202084493806265, pvalue=0.025855518458480154)

Correlation with F1
r PearsonRResult(statistic=-0.08170973609110323, pvalue=8.837105474026139e-07)
rho SignificanceResult(statistic=-0.0454258158435632, pvalue=0.006337358238047803)
tau SignificanceResult(statistic=-0.03458717356972709, pvalue=0.006130895168850152)


In [305]:
import pyterrier as pt

sparse_index = pt.Artifact.from_hf('pyterrier/ragwiki-terrier')

In [351]:
nq_index_ref = pt.IndexFactory.of('/mnt/indices/BEIR/nq/nq_sparseIndex')

nq_index_ref.getCollectionStatistics().getNumberOfDocuments()

22:37:31.441 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 660.3 MiB of memory would be required.


2681468

In [309]:
index_path ='/mnt/indices/msmarco-passage.terrier/'
index_ref = pt.IndexRef.of(index_path)
index = pt.IndexFactory.of(index_ref)
DOC_NUM = index.getCollectionStatistics().getNumberOfDocuments()

22:15:36.045 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1.9 GiB of memory would be required.


In [331]:
type(sparse_index.path)

pathlib.PosixPath

In [341]:
import torch

torch.cuda.empty_cache()